# Seguimiento y caracterización de satélites
Informe de la etapa 1 

Integrantes: Joaquin Diaz y Leonel Relmo 

In [1]:
from datetime import datetime, timezone

from funciones_base import epoch_medio
from catalogo import leer_satelites, contar_por_grupo, lanzamientos_por_anio
from satelite import crear_satelites
from analisis import (contar_por_tipo, mas_bajo, mas_alto, top_n,
                      en_zona, ordenar_por, resumen_por_grupo)

## 1. Carga de datos

In [2]:
registros = leer_satelites("datos/satelites.json")
fecha_datos = epoch_medio(registros)
ahora = datetime.now(timezone.utc)
dias = (ahora - fecha_datos).days

print(f"Satélites en el archivo: {len(registros)}")
print(f"Fecha de los datos (EPOCH medio): {fecha_datos:%d/%m/%Y}")
print(f"Momento del cálculo: {ahora:%d/%m/%Y %H:%M} UTC")
print(f"Días transcurridos desde los datos: {dias}")

sats = crear_satelites(registros, ahora)
descartados = len(registros) - len(sats)

print(f"Satélites creados: {len(sats)}")
print(f"Satélites descartados: {descartados}")

Satélites en el archivo: 16822
Fecha de los datos (EPOCH medio): 27/10/2026
Momento del cálculo: 29/10/2026 16:29 UTC
Días transcurridos desde los datos: 2
Satélites creados: 16822
Satélites descartados: 0


El archivo `satelites.json` contiene 16.822 satélites activos, tomados del catálogo público de CelesTrak. Los datos describen la órbita de cada satélite en un momento determinado (su `EPOCH`), y la fecha media de todos ellos es el 27/10/2026. Las posiciones se calcularon para el momento en que se ejecutó el notebook, 29/10/2026, es decir, 2 días después de la fecha de los datos.

Esa diferencia importa: la órbita real va cambiando de a poco, y el cálculo se aleja de la realidad a medida que el instante elegido se separa del `EPOCH`. Con pocos días el error es pequeño, pero crece con el tiempo.

`crear_satelites` descarta los satélites cuya altitud calculada es menor a 100 km, que son los que ya cayeron a la atmósfera y dan posiciones sin sentido. En esta corrida no descartó ninguno: los 16.822 satélites quedaron en la lista. Esto se debe a que los datos son muy recientes; con una fecha de cálculo más lejana al `EPOCH`, la cantidad de descartados aumenta.

## 2. Grupos

In [3]:
conteo_grupos = contar_por_grupo(registros)
for g, cantidad in sorted(conteo_grupos.items(), key=lambda par: par[1], reverse=True):
    print(f'{g}: {cantidad}')

porcentaje_starlink = conteo_grupos["Starlink"] / len(registros) * 100
print(f"\nStarlink es el {porcentaje_starlink:.1f}% del total")

Starlink: 11160
Otros: 4152
OneWeb: 651
Kuiper: 391
Qianfan: 257
Iridium: 80
BeiDou: 55
GPS: 40
Globalstar: 36

Starlink es el 66.3% del total


Los satélites se agruparon según el prefijo de su nombre. Starlink domina ampliamente el catálogo: tiene 11.160 satélites, es decir, el 66,3 % del total. Le sigue la categoría "Otros", con 4.152 satélites (24,7 %), que reúne a todos los que no pertenecen a ninguna de las constelaciones de la tabla (estaciones espaciales, satélites científicos, de observación, de comunicaciones, etc.). Después aparecen OneWeb (651), Kuiper (391) y Qianfan (257), y por último las constelaciones más chicas: Iridium (80), BeiDou (55), GPS (40) y Globalstar (36).

Las cuatro megaconstelaciones de internet (Starlink, OneWeb, Kuiper y Qianfan) suman 12.459 satélites, cerca del 74 % del total. En cambio, las constelaciones de navegación (GPS y BeiDou) tienen pocos satélites cada una.

## 3. Lanzamientos por año

In [4]:
por_anio = lanzamientos_por_anio(registros)
anios = [a for a in sorted(por_anio) if a >= 2015]
for a in anios:
    print(f'{a}: {por_anio[a]}')

anio_max = max(anios, key=lambda a: por_anio[a])
print(f"\nAño con más lanzamientos: {anio_max} ({por_anio[anio_max]})")

2015: 96
2016: 84
2017: 136
2018: 193
2019: 144
2020: 354
2021: 987
2022: 1715
2023: 2260
2024: 2428
2025: 4289
2026: 3421

Año con más lanzamientos: 2025 (4289)


La cantidad de satélites activos según su año de lanzamiento muestra un crecimiento muy fuerte en la última década. Entre 2015 y 2019 los valores son bajos y bastante estables (entre 84 y 193 por año). A partir de 2020 la cantidad empieza a subir y en 2021 casi se triplica respecto del año anterior (de 354 a 987). Después sigue aumentando año a año: 1.715 en 2022, 2.260 en 2023 y 2.428 en 2024.

El año con más lanzamientos fue 2025, con 4.289 satélites, casi el doble que en 2024. En 2026, que todavía no terminó, ya hay 3.421. Entre 2021 y 2026 se concentran 15.100 de los 16.822 satélites del archivo, cerca del 90 % del total. Este salto coincide con el despliegue de las megaconstelaciones de internet vistas en la sección anterior.

## 4. Tipos de órbita


In [5]:
tipos = contar_por_tipo(sats)
for tipo, cantidad in tipos.items():
    print(f"{tipo}: {cantidad} ({cantidad / len(sats) * 100:.1f}%)")

LEO: 16012 (95.2%)
MEO: 199 (1.2%)
GEO: 592 (3.5%)
HEO: 19 (0.1%)


Según su altitud, los satélites se clasificaron en cuatro tipos de órbita: LEO (menos de 2.000 km), MEO (de 2.000 km a menos de 34.000 km), GEO (de 34.000 km a 37.000 km) y HEO (más de 37.000 km).

La órbita que predomina ampliamente es la LEO, con 16.012 satélites (95,2 %). Le sigue la GEO, con 592 satélites (3,5 %), luego la MEO, con 199 (1,2 %), y por último la HEO, con solo 19 (0,1 %).

Este resultado es coherente con lo visto en la sección 2: las megaconstelaciones de internet (Starlink, OneWeb, Kuiper y Qianfan) suman cerca del 74 % del catálogo y orbitan a baja altura. La órbita GEO, en cambio, es una franja muy específica, la de los satélites que permanecen fijos sobre un mismo punto de la Tierra, y se usa sobre todo para comunicaciones y meteorología.

## 5. Extremos 

In [6]:
print("Satélite más bajo:", mas_bajo(sats))
print("Satélite más alto:", mas_alto(sats))

print("\nTop 10 más altos:")
for s in top_n(sats, 10):
    print(s)

print("\nTop 10 más bajos:")
for s in top_n(sats, 10, mas_altos=False):
    print(s)

Satélite más bajo: STARLINK-1036 | LEO | 117.7 km
Satélite más alto: CXO | HEO | 133250.8 km

Top 10 más altos:
CXO | HEO | 133250.8 km
SMILE | HEO | 84522.7 km
THEMIS E | HEO | 77586.0 km
THEMIS D | HEO | 61999.1 km
XMM-NEWTON | HEO | 55303.9 km
2026-167B | HEO | 47147.1 km
PROBA-3 OSC | HEO | 46571.9 km
PROBA-3 CSC | HEO | 46571.4 km
COSMOS 2590 | HEO | 43772.0 km
ASBM-2 | HEO | 43433.5 km

Top 10 más bajos:
STARLINK-1036 | LEO | 117.7 km
IPERDRONE.0 | LEO | 118.7 km
RASSVET-3 23 | LEO | 122.4 km
RASSVET-3 17 | LEO | 186.8 km
STARLINK-1309 | LEO | 203.7 km
STARLINK-1283 | LEO | 215.0 km
HYPSO-1 | LEO | 221.3 km
STARLINK-3306 | LEO | 223.8 km
NAPA-1 | LEO | 227.4 km
STARLINK-1327 | LEO | 229.0 km


El satélite más bajo de la lista es STARLINK-1036, a 117,7 km de altura. El más alto es CXO, a 133.250,8 km: está a una distancia de la Tierra mayor que la de la Luna a la mitad de su recorrido, y por eso es de tipo HEO.

Los diez satélites más altos son todos HEO, con altitudes entre 43.433 km (ASBM-2) y los 133.250 km de CXO. Entre ellos hay observatorios y misiones científicas como XMM-NEWTON, THEMIS D y E, SMILE y PROBA-3. Los diez más bajos son todos LEO, con altitudes entre 117,7 y 229,0 km, y cinco de ellos son satélites Starlink.

Ningún satélite quedó por debajo de los 100 km, lo que confirma que el filtro de `crear_satelites` funciona. Que los más bajos estén tan cerca de ese límite sugiere que son satélites que están perdiendo altura y se acercan al final de su vida en órbita.

## 6. Comparación por grupos

In [7]:
resumen = resumen_por_grupo(sats)

print(f"{'Grupo':<12}{'Cantidad':>10}{'Mínima (km)':>14}{'Media (km)':>13}{'Máxima (km)':>14}")
for g, datos in sorted(resumen.items(), key=lambda par: par[1]["cantidad"], reverse=True):
    print(f"{g:<12}{datos['cantidad']:>10}{datos['minima']:>14.1f}"
          f"{datos['media']:>13.1f}{datos['maxima']:>14.1f}")

Grupo         Cantidad   Mínima (km)   Media (km)   Máxima (km)
Starlink         11160         117.7        467.8         596.0
Otros             4152         118.7       6188.9      133250.8
OneWeb             651         518.3       1209.4        1251.2
Kuiper             391         442.7        633.2         657.1
Qianfan            257         788.0       1044.3        1139.3
Iridium             80         633.9        777.4         807.9
BeiDou              55       21506.6      27479.1       35992.0
GPS                 40       19773.1      20211.6       21027.2
Globalstar          36         680.9       1297.4        1831.0


La tabla compara la altitud de cada grupo. Hay dos familias bien diferenciadas.

Constelaciones de órbita baja. Starlink es la más baja, con una altitud media de 467,8 km, seguida por Kuiper (633,2 km), Iridium (777,4 km), Qianfan (1.044,3 km), OneWeb (1.209,4 km) y Globalstar (1.297,4 km). Todas están por debajo de los 2.000 km, es decir, en órbita LEO. Iridium (entre 633,9 y 807,9 km) y Kuiper (entre 442,7 y 657,1 km) tienen una dispersión chica, con satélites muy concentrados a una misma altura, mientras que Globalstar es la de mayor rango entre las LEO, de 680,9 a 1.831,0 km. Starlink también incluye el valor mínimo de todo el catálogo (117,7 km), muy por debajo de su media.

Constelaciones de navegación. BeiDou y GPS orbitan mucho más arriba. GPS tiene una media de 20.211,6 km y una dispersión muy baja (entre 19.773,1 y 21.027,2 km), lo que indica que todos sus satélites están a una altura casi idéntica, en órbita MEO. BeiDou tiene una media de 27.479,1 km y un rango enorme, de 21.506,6 a 35.992,0 km: su máximo cae dentro del intervalo de órbita GEO (34.000 a 37.000 km), así que esta constelación mezcla satélites MEO y GEO.

El grupo "Otros" es el más heterogéneo: va de 118,7 km a 133.250,8 km y su media (6.188,9 km) no representa bien a ningún satélite en particular, porque mezcla satélites de órbita baja con otros muy lejanos.

## 7. Bariloche

In [8]:
BARILOCHE = (-76.30, -46.13, -66.30, -36.13)   # lon_min, lat_min, lon_max, lat_max

sobre_bariloche = en_zona(sats, BARILOCHE)
print(f"Satélites sobre Bariloche: {len(sobre_bariloche)}\n")

for s in ordenar_por(sobre_bariloche, "altitud_km"):
    print(s)

Satélites sobre Bariloche: 40

STARLINK-4734 | LEO | 319.3 km
STARLINK-30235 | LEO | 340.7 km
STARLINK-11660 [DTC] | LEO | 362.2 km
STARLINK-11706 [DTC] | LEO | 363.0 km
STARLINK-11595 [DTC] | LEO | 364.3 km
STARLINK-11474 [DTC] | LEO | 370.0 km
STARLINK-38368 | LEO | 397.1 km
STARLINK-32801 | LEO | 470.1 km
STARLINK-37082 | LEO | 474.0 km
STARLINK-30703 | LEO | 475.0 km
STARLINK-34853 | LEO | 475.9 km
STARLINK-5259 | LEO | 476.6 km
STARLINK-3270 | LEO | 476.6 km
STARLINK-5179 | LEO | 477.0 km
STARLINK-36671 | LEO | 477.9 km
STARLINK-30930 | LEO | 478.4 km
STARLINK-35114 | LEO | 479.3 km
STARLINK-5176 | LEO | 485.4 km
STARLINK-31087 | LEO | 492.2 km
STARLINK-32786 | LEO | 492.5 km
STARLINK-30307 | LEO | 492.7 km
STARLINK-31188 | LEO | 493.2 km
STARLINK-34168 | LEO | 493.5 km
STARLINK-6090 | LEO | 494.2 km
STARLINK-34271 | LEO | 494.3 km
STARLINK-36493 | LEO | 494.3 km
STARLINK-5339 | LEO | 494.3 km
STARLINK-31805 | LEO | 494.7 km
STARLINK-31815 | LEO | 494.8 km
STARLINK-32396 | LEO | 4

En el momento del cálculo (29/10/2026, 16:29 UTC) había 40 satélites sobre la zona de Bariloche, definida por el rectángulo (-76,30; -46,13; -66,30; -36,13). La lista está ordenada de menor a mayor altitud.

Casi todos son LEO, con altitudes que van desde 319,3 km (STARLINK-4734) hasta unos 1.510 km (COSMOS 2498). Los más bajos y la mayor parte del listado son satélites Starlink, concentrados entre los 320 y los 500 km, lo que coincide con su altitud media vista en la sección 6. También aparecen satélites de Qianfan y OneWeb, a más de 1.000 km. El único que no es LEO es POLAR, de órbita MEO, a 20.144,7 km.

Este número no es fijo: los satélites LEO dan una vuelta a la Tierra en unos 90 minutos y se desplazan constantemente, así que la cantidad sobre una zona cambia con la hora del cálculo. Se vería distinto si el notebook se ejecutara en otro momento.